In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import joblib

from surprise import Dataset
from surprise import Reader
from surprise import SVD

from surprise.model_selection import train_test_split

from surprise import accuracy

In [4]:
learning_history = pd.read_csv(
    "/content/drive/MyDrive/learning_history.csv"
)

In [5]:
learning_history.head()

,intern_id,course_id,rating,education,completion_status
0,11391,AAA,4,HE Qualification,Pass
1,11391,AAA,4,HE Qualification,Pass
2,28400,AAA,4,HE Qualification,Pass
3,28400,AAA,4,HE Qualification,Pass
4,31604,AAA,4,A Level or Equivalent,Pass


In [6]:
print(learning_history.shape)

(660484, 5)


In [7]:
learning_history.isnull().sum()

,0
intern_id,0
course_id,0
rating,0
education,0
completion_status,0


In [8]:
ratings = learning_history[['intern_id','course_id','rating']]

In [9]:
reader = Reader(rating_scale=(1,5))

In [10]:
data = Dataset.load_from_df(
    ratings,
    reader
)

In [11]:
trainset,testset=train_test_split(
    data,
    test_size=0.20,
    random_state=42
)

In [12]:
print("Training Users :",trainset.n_users)
print("Training Courses :",trainset.n_items)

Training Users : 23348
Training Courses : 7


In [13]:
model = SVD(
    n_factors=100,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

In [14]:
print(model)

In [15]:
model.fit(trainset)

In [16]:
predictions=model.test(testset)

In [17]:
rmse=accuracy.rmse(predictions)

RMSE: 0.8496


In [18]:
mae=accuracy.mae(predictions)

MAE:  0.6851


In [19]:
print("RMSE :",rmse)
print("MAE :",mae)

RMSE : 0.8495590611601236
MAE : 0.6850798462425384


In [20]:
save_path="/content/drive/MyDrive/svd_model.pkl"

joblib.dump(model,save_path)

print("Model Saved Successfully")

Model Saved Successfully


In [21]:
from google.colab import files

files.download(save_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
intern_id=11391

In [23]:
all_courses=learning_history['course_id'].unique()

taken_courses=learning_history[
learning_history['intern_id']==intern_id
]['course_id'].values

recommendations=[]

for course in all_courses:

    if course not in taken_courses:

        pred=model.predict(intern_id,course)

        recommendations.append((course,pred.est))

In [24]:
recommendations=sorted(
recommendations,
key=lambda x:x[1],
reverse=True
)

In [25]:
top5=recommendations[:5]

for course,score in top5:
    print(course,score)

EEE 4.550764427254696
FFF 4.302421125824565
BBB 3.88745627617223
GGG 3.789127993720691
CCC 3.429188852596227


In [26]:
recommendation_df=pd.DataFrame(
top5,
columns=['Course','Predicted Rating']
)

recommendation_df.to_csv(
"/content/drive/MyDrive/recommendations.csv",
index=False
)

In [27]:
recommendation_df

,Course,Predicted Rating
0,EEE,4.550764
1,FFF,4.302421
2,BBB,3.887456
3,GGG,3.789128
4,CCC,3.429189


In [28]:
files.download(
"/content/drive/MyDrive/recommendations.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>